In [ ]:
import pandas as pd
from pathlib import Path

PROCESSED_DATA_DIR = Path("../data/processed")

In [ ]:
df = pd.read_parquet(PROCESSED_DATA_DIR / "strandings_with_weather.parquet")

In [ ]:
min(df['mms_observation_dt'])

In [ ]:
cycle = 29.530588
first_new_moon = pd.Timestamp("2016-01-10")
def moon_cycle(age):
    if age < 1.84566:
        return "New Moon"
    elif age < 5.53699:
        return "Waxing Crescent"
    elif age < 9.22831:
        return "First Quarter"
    elif age < 12.91963:
        return "Waxing Gibbous"
    elif age < 16.61096:
        return "Full Moon"
    elif age < 20.30228:
        return "Waning Gibbous"
    elif age < 23.99361:
        return "Last Quarter"
    elif age < 27.68493:
        return "Waning Crescent"
    else:
        return "New Moon"

In [ ]:
def add_moon_cycle(df, mms_observation_dt):
    df = df.copy()
    df[mms_observation_dt] = pd.to_datetime(df[mms_observation_dt])

    days_since = (df[mms_observation_dt] - first_new_moon).dt.days
    moon_age = days_since % cycle

    df["moon_age"] = moon_age
    df["moon_phase"] = moon_age.apply(moon_cycle)

    return df

In [ ]:
df

In [ ]:
df = add_moon_cycle(df, "mms_observation_dt")

In [ ]:
df

In [ ]:
from se_coast_strandings.contextual_data.lunar_phases import add_moon_features

# Re-apply using the module function for consistency
df = add_moon_features(df, date_col="mms_observation_dt")
df.to_parquet(PROCESSED_DATA_DIR / "strandings_with_moon.parquet", index=True)
print(f"Saved strandings_with_moon.parquet: {len(df)} rows")